# 01 — Reestructuracion en modelo dimensional simplificado

**Objetivo especifico que responde:** OE-02 del anteproyecto (`InformesRev/AnteproyectoAndres.md`, numeral 1.4.2): *"Estructurar los conjuntos de datos de SECOP relacionados con obra publica (procesos, contratos y proveedores), definiendo el modelo de datos analitico y las reglas de integracion necesarias para su relacionamiento consistente."*

**Insumo:** `entregables/04-datasets/contratos_adiciones_obra.parquet` (integracion primaria contratos + adiciones, producida por el pipeline compartido `notebooks/05_integracion_contratos_adiciones.ipynb`).

**Camino de decision aplicado:** Camino B de `MATRIZ_OBJETIVOS_CODIGO.md` (seccion 3, OE-02): *modelo dimensional simplificado* sobre archivos Parquet (tabla de hechos + dimensiones, con diagrama y diccionario de convenciones), en lugar de un modelo estrella formal materializado en un motor relacional. Esta decision fue aprobada explicitamente como via de cierre de la brecha de OE-02 en la matriz de objetivos, dado el tiempo disponible y la ausencia de un motor de base de datos definido en el anteproyecto original.

## Advertencia de dependencia de autoria del dato de entrada (obligatoria, ver `PROMPT_EDA_EXTENSO_OE02_OE03.md`, regla 4)

El dataset de entrada de este notebook (`contratos_adiciones_obra.parquet`) fue producido por el pipeline compartido (`notebooks/00` a `06`), cuya autoria **aun no ha sido confirmada por el director** (ver `MATRIZ_OBJETIVOS_CODIGO.md`, plan de accion, tarea de prioridad 1). Este notebook es independiente en **codigo y logica analitica** — ninguna linea de este notebook fue copiada, adaptada ni inspirada en `notebooks/07_depuracion_variables_modelo.ipynb` ni en `notebooks/08_integracion_rup_no_consorcios.ipynb` (ambos clasificados como "CODIGO DEL COMPANERO - EXCLUIR")—, pero **no elimina** la necesidad de esa confirmacion respecto al dato crudo integrado. Este punto debe quedar resuelto con el director antes de presentar este trabajo como evidencia final.

**Columnas excluidas deliberadamente de este notebook** por ser derivadas especificamente por el companero para su modelado predictivo (presentes en el parquet de entrada pero fuera de uso aqui): `tiempo`, `presupuesto`, `alcance`, `sobrecosto_ratio`, `duracion_planificada_dias`, `ratio_extension_duracion`, `dias_hasta_primera_adicion`, `ventana_adiciones_dias`, `log_valor_contrato`, y todas las columnas `rup_*` / `pliego_*`. Se usan unicamente las columnas crudas o agregadas por el pipeline compartido de extraccion/integracion (no las derivadas de depuracion para modelado).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

REPO = Path(".").resolve().parents[1] if Path(".").resolve().name == "notebooks" else Path(".").resolve()
BASE_DATASET = REPO / "entregables" / "04-datasets" / "contratos_adiciones_obra.parquet"
OUT_DIR = REPO / "analitica-descriptiva-andres" / "data" / "modelo_dimensional"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(BASE_DATASET)
print("Dataset base cargado:", df.shape)
print("Columnas totales:", len(df.columns))


Dataset base cargado: (48331, 125)
Columnas totales: 125


## 1. Criterio de diseno del modelo dimensional

Se adopta un enfoque de **grano por contrato**: cada fila de la tabla de hechos representa un contrato de obra publica unico (`id_contrato`). Las dimensiones se separan por tema, siguiendo el criterio clasico de modelado dimensional (Kimball & Ross, 2013, ya citado en el marco tecnologico del anteproyecto, numeral 2.2.5):

- **`dim_modalidad`**: como se selecciono al proveedor (mecanismo de seleccion).
- **`dim_proveedor`**: quien ejecuta el contrato (tipo de documento, si es grupo/consorcio, si es pyme, tipo de contratista derivado).
- **`dim_entidad`**: quien contrata (entidad, sector, rama, nivel de centralizacion).
- **`dim_territorio`**: donde ocurre el contrato (departamento, ciudad, orden — nacional/territorial).
- **`dim_proceso`**: atributos propios del proceso de compra que no pertenecen a ninguna de las dimensiones anteriores.

La tabla de hechos conserva las medidas cuantitativas (valor del contrato, valor pagado, fechas, conteos de modificaciones) y las llaves foraneas hacia cada dimension.

In [2]:
# --- dim_modalidad ---
dim_modalidad = (
    df[['modalidad_de_contratacion']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_modalidad['modalidad_id'] = dim_modalidad.index + 1
dim_modalidad = dim_modalidad[['modalidad_id', 'modalidad_de_contratacion']]
print("dim_modalidad:", dim_modalidad.shape)
dim_modalidad.head(10)


dim_modalidad: (9, 2)


,modalidad_id,modalidad_de_contratacion
0,1,Selección Abreviada de Menor Cuantía
1,2,Licitación pública Obra Publica
2,3,Mínima cuantía
3,4,Contratación régimen especial
4,5,Seleccion Abreviada Menor Cuantia Sin Manifest...
5,6,Licitación pública
6,7,Contratación régimen especial (con ofertas)
7,8,Contratación directa
8,9,Contratación Directa (con ofertas)


In [3]:
# --- dim_proveedor ---
# Derivacion independiente de "tipo_contratista" (NO proviene de ningun notebook del companero):
# es_grupo == 'Si'  -> Consorcio / Union Temporal
# es_grupo == 'No' y tipodocproveedor == 'NIT' -> Persona Juridica
# es_grupo == 'No' y tipodocproveedor en documentos de persona natural -> Persona Natural
# cualquier otra combinacion -> No definido

docs_persona_natural = ['Cédula de Ciudadanía', 'Cédula de Extranjería', 'Tarjeta de Identidad',
                          'Permiso especial de permanencia']

def derivar_tipo_contratista(row):
    if row['es_grupo'] == 'Si':
        return 'Consorcio / Union Temporal'
    if row['tipodocproveedor'] == 'NIT':
        return 'Persona Juridica'
    if row['tipodocproveedor'] in docs_persona_natural:
        return 'Persona Natural'
    return 'No definido'

dim_proveedor = (
    df[['codigo_proveedor', 'tipodocproveedor', 'es_grupo', 'es_pyme']]
    .drop_duplicates(subset=['codigo_proveedor'])
    .reset_index(drop=True)
)
dim_proveedor['tipo_contratista'] = dim_proveedor.apply(derivar_tipo_contratista, axis=1)
print("dim_proveedor:", dim_proveedor.shape)
print(dim_proveedor['tipo_contratista'].value_counts(dropna=False))


dim_proveedor: (23192, 5)
tipo_contratista
Persona Juridica              11254
Consorcio / Union Temporal     9378
Persona Natural                2461
No definido                      99
Name: count, dtype: int64


In [4]:
# --- dim_entidad ---
dim_entidad = (
    df[['codigo_entidad', 'nombre_entidad', 'nit_entidad', 'sector', 'rama', 'entidad_centralizada']]
    .drop_duplicates(subset=['codigo_entidad'])
    .reset_index(drop=True)
)
print("dim_entidad:", dim_entidad.shape)
dim_entidad.head(5)


dim_entidad: (2534, 6)


,codigo_entidad,nombre_entidad,nit_entidad,sector,rama,entidad_centralizada
0,701699357,POLICÍA METROPOLITANA DE MANIZALES,900805219,defensa,Ejecutivo,Centralizada
1,701189037,ARMADA NACIONAL - BASE NAVAL No.1 ARC BOLÍVAR,800141645,defensa,Ejecutivo,Centralizada
2,700676059,INVIAS,800215807,Transporte,Ejecutivo,Centralizada
3,702950643,CENTRAL ADMINISTRATIVA Y CONTABLE CENAC PERSONAL,900385079,defensa,Ejecutivo,Descentralizada
4,701821332,COMANDO AÉREO DE COMBATE No. 1,800141624,No aplica/No pertenece,Corporación Autónoma,Centralizada


In [5]:
# --- dim_territorio ---
# Llave subrogada por combinacion unica de departamento + ciudad + orden
dim_territorio = (
    df[['departamento', 'ciudad', 'orden']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_territorio['territorio_id'] = dim_territorio.index + 1
dim_territorio = dim_territorio[['territorio_id', 'departamento', 'ciudad', 'orden']]
print("dim_territorio:", dim_territorio.shape)
dim_territorio.head(5)


dim_territorio: (848, 4)


,territorio_id,departamento,ciudad,orden
0,1,Caldas,Manizales,Nacional
1,2,Bolívar,Cartagena,Nacional
2,3,Distrito Capital de Bogotá,Bogotá,Nacional
3,4,Distrito Capital de Bogotá,Bogotá,Territorial
4,5,No Definido,No Definido,Nacional


In [6]:
# --- dim_proceso ---
dim_proceso = (
    df[['proceso_de_compra', 'codigo_de_categoria_principal', 'justificacion_modalidad_de']]
    .drop_duplicates(subset=['proceso_de_compra'])
    .reset_index(drop=True)
)
print("dim_proceso:", dim_proceso.shape)
dim_proceso.head(5)


dim_proceso: (43451, 3)


,proceso_de_compra,codigo_de_categoria_principal,justificacion_modalidad_de
0,CO1.BDOS.825935,V1.72101500,Presupuesto menor al 10% de la Menor Cuantía
1,CO1.BDOS.813978,V1.72101500,Presupuesto menor al 10% de la Menor Cuantía
2,CO1.BDOS.797027,V1.72141000,Article30_1993
3,CO1.BDOS.853412,V1.49201500,Presupuesto inferior al 10% de la menor cuantía
4,CO1.BDOS.825838,V1.72102900,Presupuesto menor al 10% de la Menor Cuantía


In [7]:
# --- hechos_contratos ---
hechos = df[[
    'id_contrato', 'proceso_de_compra', 'codigo_proveedor', 'codigo_entidad',
    'departamento', 'ciudad', 'orden', 'modalidad_de_contratacion',
    'fecha_de_firma', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato',
    'duraci_n_del_contrato', 'valor_del_contrato', 'valor_pagado',
    'dias_adicionados', 'n_adicion_valor', 'n_modificaciones_total',
    'tiene_suspension', 'tiene_cesion', 'tiene_conclusion',
    'estado_contrato', 'liquidaci_n',
]].copy()

# Reemplazar atributos por llaves subrogadas/naturales de cada dimension
hechos = hechos.merge(dim_modalidad, on='modalidad_de_contratacion', how='left')
hechos = hechos.merge(dim_territorio, on=['departamento', 'ciudad', 'orden'], how='left')

hechos_contratos = hechos[[
    'id_contrato', 'proceso_de_compra', 'codigo_proveedor', 'codigo_entidad',
    'territorio_id', 'modalidad_id',
    'fecha_de_firma', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato',
    'duraci_n_del_contrato', 'valor_del_contrato', 'valor_pagado',
    'dias_adicionados', 'n_adicion_valor', 'n_modificaciones_total',
    'tiene_suspension', 'tiene_cesion', 'tiene_conclusion',
    'estado_contrato', 'liquidaci_n',
]].copy()

print("hechos_contratos:", hechos_contratos.shape)
hechos_contratos.head(3)


hechos_contratos: (48331, 20)


,id_contrato,proceso_de_compra,codigo_proveedor,codigo_entidad,territorio_id,modalidad_id,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,duraci_n_del_contrato,valor_del_contrato,valor_pagado,dias_adicionados,n_adicion_valor,n_modificaciones_total,tiene_suspension,tiene_cesion,tiene_conclusion,estado_contrato,liquidaci_n
0,CO1.PCCNTR.1001032,CO1.BDOS.825935,701107773,701699357,1,1,2019-06-25,2019-06-25,2019-12-29,Dia(s),997710826,0,0,0,4,0,0,0,terminado,Si
1,CO1.PCCNTR.1001046,CO1.BDOS.813978,701794463,701189037,2,1,2019-06-21,2019-06-25,2019-12-16,Dia(s),756004228,161267354,0,0,4,0,0,1,Cerrado,Si
2,CO1.PCCNTR.1001111,CO1.BDOS.797027,704212661,700676059,3,2,2019-08-14,2019-07-15,2019-11-15,Dia(s),926429147,0,0,0,2,0,0,0,terminado,No


In [8]:
# --- Verificacion de integridad referencial ---
checks = {
    'proceso_de_compra -> dim_proceso': hechos_contratos['proceso_de_compra'].isin(dim_proceso['proceso_de_compra']).mean(),
    'codigo_proveedor -> dim_proveedor': hechos_contratos['codigo_proveedor'].isin(dim_proveedor['codigo_proveedor']).mean(),
    'codigo_entidad -> dim_entidad': hechos_contratos['codigo_entidad'].isin(dim_entidad['codigo_entidad']).mean(),
    'territorio_id -> dim_territorio': hechos_contratos['territorio_id'].isin(dim_territorio['territorio_id']).mean(),
    'modalidad_id -> dim_modalidad': hechos_contratos['modalidad_id'].isin(dim_modalidad['modalidad_id']).mean(),
}
for k, v in checks.items():
    print(f"{k}: {v*100:.2f}% de coincidencia")

assert hechos_contratos['id_contrato'].is_unique, "id_contrato no es unico en hechos_contratos"
print("\nGrano verificado: id_contrato es unico en hechos_contratos (", hechos_contratos['id_contrato'].nunique(), "valores unicos ).")


proceso_de_compra -> dim_proceso: 100.00% de coincidencia
codigo_proveedor -> dim_proveedor: 100.00% de coincidencia
codigo_entidad -> dim_entidad: 100.00% de coincidencia
territorio_id -> dim_territorio: 100.00% de coincidencia
modalidad_id -> dim_modalidad: 100.00% de coincidencia



Grano verificado: id_contrato es unico en hechos_contratos ( 48331 valores unicos ).


In [9]:
# --- Persistencia del modelo dimensional ---
hechos_contratos.to_parquet(OUT_DIR / "hechos_contratos.parquet", index=False)
dim_proceso.to_parquet(OUT_DIR / "dim_proceso.parquet", index=False)
dim_proveedor.to_parquet(OUT_DIR / "dim_proveedor.parquet", index=False)
dim_entidad.to_parquet(OUT_DIR / "dim_entidad.parquet", index=False)
dim_territorio.to_parquet(OUT_DIR / "dim_territorio.parquet", index=False)
dim_modalidad.to_parquet(OUT_DIR / "dim_modalidad.parquet", index=False)

for f in OUT_DIR.glob("*.parquet"):
    print(f.name, "-", pd.read_parquet(f).shape)


dim_proceso.parquet - (43451, 3)


dim_entidad.parquet - (2534, 6)
dim_territorio.parquet - (848, 4)
dim_modalidad.parquet - (9, 2)
dim_proveedor.parquet - (23192, 5)
hechos_contratos.parquet - (48331, 20)


## 2. Diagrama del esquema dimensional

```mermaid
erDiagram
    HECHOS_CONTRATOS }o--|| DIM_PROCESO : proceso_de_compra
    HECHOS_CONTRATOS }o--|| DIM_PROVEEDOR : codigo_proveedor
    HECHOS_CONTRATOS }o--|| DIM_ENTIDAD : codigo_entidad
    HECHOS_CONTRATOS }o--|| DIM_TERRITORIO : territorio_id
    HECHOS_CONTRATOS }o--|| DIM_MODALIDAD : modalidad_id

    HECHOS_CONTRATOS {
        string id_contrato PK
        string proceso_de_compra FK
        string codigo_proveedor FK
        string codigo_entidad FK
        int territorio_id FK
        int modalidad_id FK
        date fecha_de_firma
        date fecha_de_inicio_del_contrato
        date fecha_de_fin_del_contrato
        string duraci_n_del_contrato
        float valor_del_contrato
        float valor_pagado
        int dias_adicionados
        int n_adicion_valor
        int n_modificaciones_total
        bool tiene_suspension
        bool tiene_cesion
        bool tiene_conclusion
        string estado_contrato
        string liquidaci_n
    }
    DIM_PROCESO {
        string proceso_de_compra PK
        string codigo_de_categoria_principal
        string justificacion_modalidad_de
    }
    DIM_PROVEEDOR {
        string codigo_proveedor PK
        string tipodocproveedor
        string es_grupo
        string es_pyme
        string tipo_contratista
    }
    DIM_ENTIDAD {
        string codigo_entidad PK
        string nombre_entidad
        string nit_entidad
        string sector
        string rama
        string entidad_centralizada
    }
    DIM_TERRITORIO {
        int territorio_id PK
        string departamento
        string ciudad
        string orden
    }
    DIM_MODALIDAD {
        int modalidad_id PK
        string modalidad_de_contratacion
    }
```

## 3. Diccionario de convenciones

| Tabla | Grano | Llave primaria | Llave(s) foranea(s) |
|---|---|---|---|
| `hechos_contratos` | 1 fila = 1 contrato de obra publica unico | `id_contrato` | `proceso_de_compra`, `codigo_proveedor`, `codigo_entidad`, `territorio_id`, `modalidad_id` |
| `dim_proceso` | 1 fila = 1 proceso de compra | `proceso_de_compra` | — |
| `dim_proveedor` | 1 fila = 1 proveedor unico | `codigo_proveedor` | — |
| `dim_entidad` | 1 fila = 1 entidad contratante unica | `codigo_entidad` | — |
| `dim_territorio` | 1 fila = 1 combinacion unica departamento-ciudad-orden | `territorio_id` (subrogada) | — |
| `dim_modalidad` | 1 fila = 1 modalidad de contratacion unica | `modalidad_id` (subrogada) | — |

## 4. Limitacion declarada

Este es un **modelo dimensional simplificado materializado sobre archivos Apache Parquet**, no un modelo estrella formal implementado en un motor de base de datos relacional (PostgreSQL, MySQL) ni analitico (DuckDB, SQLite). Se opto por esta via siguiendo el Camino B de decision ya aprobado en `MATRIZ_OBJETIVOS_CODIGO.md` (seccion 3, OE-02), dado el tiempo disponible y la ausencia de un motor de almacenamiento definido en el anteproyecto original (numeral 2.2.5, marcado alli como "no especificado"). Las relaciones entre tablas se sostienen por convencion documentada (este diccionario) y se verificaron programaticamente (celda de integridad referencial arriba), pero no estan garantizadas por restricciones `FOREIGN KEY` de un motor real.

## 5. Sintesis de hallazgos de este notebook

- Se separaron 125 columnas del dataset integrado en 1 tabla de hechos (grano contrato) y 5 tablas de dimension, con verificacion de integridad referencial superior al 99% en todas las relaciones.
- El grano de `hechos_contratos` es unico por `id_contrato` (48.331 contratos), consistente con la unidad de analisis declarada en el anteproyecto (numeral 3.5).
- Se derivo de forma independiente la variable `tipo_contratista` (Persona Natural / Persona Juridica / Consorcio-Union Temporal / No definido) a partir de `es_grupo` y `tipodocproveedor`, necesaria para contrastar la hipotesis H2 del anteproyecto en el notebook 07.